<a href="https://colab.research.google.com/github/noorxwaleed-oss/online-AI-host/blob/main/content_analyzer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install --upgrade --quiet \
    langchain-community \
    langchain-openai \
    chromadb \
    pypdf \
    pandas \
    streamlit \
    python-dotenv


In [ ]:
import os
import json
import uuid
from openai import OpenAI
from google.colab import userdata
from langchain_community.document_loaders import PyPDFLoader, WebBaseLoader
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
import bs4


In [ ]:



# =============================
# 1. Setup
# =============================
import osfrom dotenv import load_dotenvload_dotenv()client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=userdata.get("OPENROUTER_API_KEY" ),
)



In [ ]:
def load_data(source):
    if source.startswith("http" ):
        print(f"🌐 جاري تحميل الرابط...")
        loader = WebBaseLoader(
            web_path=(source,),
            bs_kwargs=dict(parse_only=bs4.SoupStrainer(name=("article", "h1", "h2", "h3", "p")))
        )
    else:
        print(f"📄 جاري تحميل ملف PDF...")
        loader = PyPDFLoader(source)
    return loader.load()


In [ ]:
import hashlib
import os

# 1. دالة الـ Embeddings (كما هي)
def get_embedding_func():
    return HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# 2. دالة إنشاء الـ Vectorstore (المعدلة لتكون أكثر استقراراً)
def create_vectorstore(chunks, embedding_func, vectorstore_path):
    print(f"🛠️ جاري إنشاء قاعدة بيانات في المجلد: {vectorstore_path}")

    # تنظيف البيانات ومنع التكرار
    ids = [str(uuid.uuid5(uuid.NAMESPACE_DNS, doc.page_content)) for doc in chunks]
    unique_ids, unique_chunks = set(), []
    for chunk, id in zip(chunks, ids):
        if id not in unique_ids:
            unique_ids.add(id)
            unique_chunks.append(chunk)

    # إنشاء قاعدة البيانات (تم حذف .persist() لتجنب أخطاء الـ Readonly في Colab)
    vectorstore = Chroma.from_documents(
        documents=unique_chunks,
        ids=list(unique_ids),
        embedding=embedding_func,
        persist_directory=vectorstore_path
    )

    return vectorstore


In [ ]:
source_input = ""
# --- الكود السحري لتوليد اسم مجلد فريد بناءً على اسم الملف ---
# نأخذ اسم الملف فقط ونحوله لـ "بصمة" (Hash) قصيرة
file_name = os.path.basename(source_input)
folder_suffix = hashlib.md5(file_name.encode()).hexdigest()[:8]
unique_path = f"vectorstore_{folder_suffix}"

# الآن نستدعي الدوال باستخدام الاسم الفريد
embedding_func = get_embedding_func()
pages = load_data(source_input)

# إنشاء أو تحميل قاعدة البيانات الفريدة
if not os.path.exists(unique_path):
    vectorstore = create_vectorstore(pages, embedding_func, unique_path)
else:
    print(f"♻️ تم العثور على ذاكرة سابقة لهذا الملف في: {unique_path}")
    vectorstore = Chroma(persist_directory=unique_path, embedding_function=embedding_func)


In [ ]:

user_language_choice = ""
def run_smart_analysis(vectorstore,target_lang):
    retriever = vectorstore.as_retriever(search_kwargs={"k": 15})
    relevant_chunks = retriever.invoke("استخرج أهم الأفكار والنتائج الرئيسية للبودكاست")
    context = "\n\n".join([chunk.page_content for chunk in relevant_chunks])

    PROMPT_TEMPLATE = """
    You are a Content Analyzer Agent. Your goal is to prepare material for a professional podcast.
    Analyze the context and extract up to 15 unique, high-value topics.

    Content: {context}

    INSTRUCTION FOR SHORT CONTENT:
    - Assess the depth of the provided 'Context'.
    - If the context is rich and long, extract 15-20 unique topics. Set "content_status" to "sufficient".
    - If the context is short or lacks depth, extract only 5 high-quality topics. Set "content_status" to "limited" and provide a brief warning message in "system_note" in {target_lang}.

    Rules:
    - LANGUAGE RULE: Analyze the provided content and provide the output in {target_lang}.
    - STRICT PROHIBITION: Do NOT use any Chinese characters (中文), even if the model defaults to it.
    - Extract 15 to 20 distinct topics.
    - Focus on facts, stories, and controversial points suitable for a 15-minute dialogue.
    - The entire output content MUST be in {target_lang}.
    - Return ONLY valid JSON.

    Format:
    {{
      "content_status": "sufficient" or "limited",
      "system_note": "A message to the user if content is limited",
      "topics": [
        {{
          "title": "Topic Title",
          "key_points": ["Point 1", "Point 2"],
          "insight": "Deep takeaway",
          "discussion_angles": ["Question for the host", "Counter-argument"]
        }}
      ]
    }}
    """

    completion = client.chat.completions.create(
         #model="qwen/qwen-2.5-72b-instruct",
         model ="meta-llama/llama-3.3-70b-instruct",
         messages=[{"role": "user", "content": PROMPT_TEMPLATE.format(context=context, target_lang=target_lang)}]

        # أزلنا response_format مؤقتاً لزيادة التوافق مع جميع الموديلات
    )


    raw_content = completion.choices[0].message.content.strip()

    # محاولة تنظيف النص إذا أضاف الموديل أي كلام خارج الـ JSON
    if "```json" in raw_content:
        raw_content = raw_content.split("```json")[1].split("```")[0].strip()
    elif "```" in raw_content:
        raw_content = raw_content.split("```")[1].split("```")[0].strip()

    try:
        return json.loads(raw_content)
    except json.JSONDecodeError:
        print("⚠️ فشل في تحويل النص إلى JSON، النص الخام المستلم:")
        print(raw_content)
        raise

In [ ]:
try:
    print("🧠 جاري تحليل المحتوى الآن... قد يستغرق ذلك بضع ثوانٍ.")
    final_analysis = run_smart_analysis(vectorstore, user_language_choice)

    print(f"✅ تم التحليل بنجاح! تم استخراج {len(final_analysis['topics'])} موضوعاً.")

    # طباعة النتيجة النهائية بشكل جميل ومنظم
    print("-" * 30)
    print(json.dumps(final_analysis, indent=2, ensure_ascii=False))
    print("-" * 30)

except Exception as e:
    print(f"❌ حدث خطأ أثناء التحليل: {e}")